## Library

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact
from scipy.integrate import solve_ivp

## Problem 01

In [ ]:
K_SE = 136
K_PE = 75
b = 50
c1 = (K_SE + K_PE) / b
c2 = K_SE / b

dt = 0.001
t = np.arange(0, 2.0, dt)

def simulate_muscle(freq):
    period = 1.0 / freq
    A_total = np.zeros_like(t)

    stim_times = np.arange(0, 1.5, period)
    for stim_t in stim_times:
        idx = t >= stim_t
        t_shift = t[idx] - stim_t
        A_twitch = 48144 * np.exp(-t_shift / 0.0326) - 45845 * np.exp(-t_shift / 0.034)
        A_total[idx] += A_twitch

    T = np.zeros_like(t)
    for i in range(len(t) - 1):
        dT = -c1 * T[i] + c2 * A_total[i]
        T[i+1] = T[i] + dT * dt

    return T

def plot_muscle(freq):
    T = simulate_muscle(freq)

    plt.figure(figsize=(10, 5))
    plt.plot(t, T, lw=2, color='darkred')
    plt.xlabel('Time (s)')
    plt.ylabel('Muscle Force T (g)')
    plt.title(f'Muscle Force at {freq:.1f} Hz')
    plt.grid(True)

    if np.max(T) > 0:
        plt.ylim(0, np.max(T) * 1.2)

    plt.show()

interact(
    plot_muscle,
    freq=widgets.FloatSlider(
        value=20.0,
        min=10.0,
        max=400.0,
        step=1.0,
        description='주파수(Hz):',
        continuous_update=True
    )
);

In [ ]:
def find_fused_tetanus(threshold_percent):
    freq = 10.0

    while freq <= 200.0:
        T = simulate_muscle(freq)
        steady_state_idx = (t >= 1.0) & (t <= 1.5)
        T_steady = T[steady_state_idx]

        T_max = np.max(T_steady)
        T_min = np.min(T_steady)
        ripple_percent = (T_max - T_min) / T_max * 100

        if ripple_percent <= threshold_percent:
            print(f"-> 필요 주파수: {freq} Hz")
            print(f"-> 최종 진동 폭: {ripple_percent:.2f}%")

            plt.figure(figsize=(10, 5))
            plt.plot(t, T, lw=2, color='darkblue')
            plt.axvspan(1.0, 1.5, color='gray', alpha=0.2, label='Steady-state Window')
            plt.title(f'Fused Tetanus Reached at {freq} Hz')
            plt.xlabel('Time (s)')
            plt.ylabel('Muscle Force T (g)')
            plt.ylim(0, np.max(T) * 1.2)
            plt.legend()
            plt.grid(True)
            plt.show()

            return freq

        freq += 1.0

    return None

target_frequency = find_fused_tetanus(1.0)

## Problem 02

In [ ]:
F_max = 45.7
l_o = 6.6
K_sh = 3.0
a1 = 0.8
a2 = 0.5
a3 = 0.43
a4 = 58.0

alphas = [0, 0.25, 0.5, 0.75, 1.0]
l_array = np.linspace(0.5 * l_o, 1.5 * l_o, 200)
v_array = np.linspace(-0.2, 0.2, 200)

def calc_F_total(alpha, l, v):
    u = 1 - 4 * ((l - l_o) / l_o)**2
    F_a = alpha * F_max * np.maximum(u, 0)

    F_p = np.zeros_like(l)
    if isinstance(l, np.ndarray):
        mask = l >= l_o
        F_p[mask] = (F_max / (np.exp(K_sh) - 1)) * (np.exp(K_sh * (l[mask] - l_o) / (0.5 * l_o)) - 1)
    else:
        if l >= l_o:
            F_p = (F_max / (np.exp(K_sh) - 1)) * (np.exp(K_sh * (l - l_o) / (0.5 * l_o)) - 1)

    return (a1 + a2 * np.arctan(a3 + a4 * v)) * (F_a + F_p)

fig = plt.figure(figsize=(18, 5))
ax1 = fig.add_subplot(1, 3, 1)

for alpha in alphas:
    F_total_iso = calc_F_total(alpha, l_array, 0.0)
    ax1.plot(l_array, F_total_iso, label=f'$\\alpha$ = {alpha}')

F_passive_only = calc_F_total(0.0, l_array, 0.0)
ax1.plot(l_array, F_passive_only, 'k--', alpha=0.5, label='Passive Only')

ax1.set_xlabel('Muscle Length $l$ (cm)')
ax1.set_ylabel('Total Force $F_{total}$ (N)')
ax1.set_title('Force-Length Curve (Isometric, $\\dot{l}=0$)')
ax1.grid(True)
ax1.legend()


ax2 = fig.add_subplot(1, 3, 2)

for alpha in alphas:
    F_total_v = calc_F_total(alpha, l_o, v_array)
    ax2.plot(v_array, F_total_v, label=f'$\\alpha$ = {alpha}')

ax2.set_xlabel('Muscle Velocity $\\dot{l}$ (m/s) \n [ -: Shortening, +: Lengthening ]')
ax2.set_ylabel('Total Force $F_{total}$ (N)')
ax2.set_title('Force-Velocity Curve (at Optimal Length $l_o$)')
ax2.grid(True)
ax2.legend()


ax3 = fig.add_subplot(1, 3, 3, projection='3d')
L_grid, V_grid = np.meshgrid(l_array, v_array)
F_total_3d = calc_F_total(1.0, L_grid, V_grid)

surf = ax3.plot_surface(L_grid, V_grid, F_total_3d, cmap='viridis', edgecolor='none', alpha=0.9)
ax3.set_xlabel('Length $l$ (cm)')
ax3.set_ylabel('Velocity $\\dot{l}$ (m/s)')
ax3.set_zlabel('Force $F_{total}$ (N)')
ax3.set_title('3D Force-Length-Velocity Curve ($\\alpha$ = 1.0)')

fig.colorbar(surf, ax=ax3, shrink=0.5, aspect=5, label='Force (N)')

plt.tight_layout()
plt.show()

## Problem 03

In [ ]:
Cm = 0.01; g_Na = 1.2; g_K = 0.36; g_l = 0.003
E_Na = 55.17; E_K = -72.14; E_l = -49.42

def alpha_n(v): return 0.01 * (v + 50) / (1 - np.exp(-(v + 50) / 10) + 1e-8)
def beta_n(v):  return 0.125 * np.exp(-(v + 60) / 80)
def alpha_m(v): return 0.1 * (v + 35) / (1 - np.exp(-(v + 35) / 10) + 1e-8)
def beta_m(v):  return 4.0 * np.exp(-0.0556 * (v + 60))
def alpha_h(v): return 0.07 * np.exp(-0.05 * (v + 60))
def beta_h(v):  return 1 / (1 + np.exp(-0.1 * (v + 30)))

def hh_model(t, y, I_inj):
    v, m, h, n = y
    dm_dt = alpha_m(v) * (1 - m) - beta_m(v) * m
    dh_dt = alpha_h(v) * (1 - h) - beta_h(v) * h
    dn_dt = alpha_n(v) * (1 - n) - beta_n(v) * n

    I_Na = g_Na * (m**3) * h * (v - E_Na)
    I_K = g_K * (n**4) * (v - E_K)
    I_L = g_l * (v - E_l)

    dv_dt = (I_inj - I_Na - I_K - I_L) / Cm
    return [dv_dt, dm_dt, dh_dt, dn_dt]

v0 = -60.0
y0 = [v0, alpha_m(v0)/(alpha_m(v0)+beta_m(v0)),
      alpha_h(v0)/(alpha_h(v0)+beta_h(v0)),
      alpha_n(v0)/(alpha_n(v0)+beta_n(v0))]
t_span = (0, 50)
t_eval = np.linspace(0, 50, 1000)

def plot_hh_simulation(I_inj):
    sol = solve_ivp(hh_model, t_span, y0, args=(I_inj,), t_eval=t_eval, method='BDF')

    plt.figure(figsize=(10, 5))
    plt.plot(sol.t, sol.y[0], 'b-', lw=2, label='Membrane Potential (v)')
    plt.axhline(-60, color='gray', linestyle='--', alpha=0.5, label='Resting Potential')
    plt.title(rf'Hodgkin-Huxley Step Response (Injected Current: {I_inj:.3f} $\mu A/cm^2$)')
    plt.xlabel('Time (ms)')
    plt.ylabel('Voltage (mV)')
    plt.ylim(-80, 60)
    plt.grid(True)
    plt.legend()
    plt.show()

interact(
    plot_hh_simulation,
    I_inj=widgets.FloatSlider(
        value=0.023,
        min=0.000,
        max=0.100,
        step=0.001,
        description='전류 입력(I):',
        readout_format='.3f',
        continuous_update=True
    )
);

## Problem 04

In [ ]:
Tr = 1.0
Ta = 12.0
b = 2.5
a12 = 1.5
a21 = 1.5
u1 = 5.0
u2 = 5.0
s1 = 0.0
s2 = 0.0

def neural_oscillator(t, state):
    x1, z1, x2, z2 = state
    y1 = max(0, x1)
    y2 = max(0, x2)
    dx1_dt = (-x1 - a12 * y2 - b * z1 + u1 + s1) / Tr
    dz1_dt = (-z1 + y1) / Ta
    dx2_dt = (-x2 - a21 * y1 - b * z2 + u2 + s2) / Tr
    dz2_dt = (-z2 + y2) / Ta
    return [dx1_dt, dz1_dt, dx2_dt, dz2_dt]

t_span = (0, 100)
t_eval = np.linspace(0, 100, 1000)

def plot_oscillator(x1_init, x2_init):
    state0 = [x1_init, 0.0, x2_init, 0.0]
    sol = solve_ivp(neural_oscillator, t_span, state0, t_eval=t_eval, method='RK45')

    y1_out = np.maximum(0, sol.y[0])
    y2_out = np.maximum(0, sol.y[2])

    plt.figure(figsize=(10, 5))
    plt.plot(sol.t, y1_out, 'r-', lw=2, label='$y_1$ (Neuron 1)')
    plt.plot(sol.t, y2_out, 'b-', lw=2, label='$y_2$ (Neuron 2)')

    plt.title(f'Neural Oscillator Output (Initial $x_1$={x1_init}, $x_2$={x2_init})')
    plt.xlabel('Time (s)')
    plt.ylabel('Output $y_i$')
    plt.ylim(-0.5, 3.5)
    plt.legend(loc='upper right')
    plt.grid(True)
    plt.show()

interact(
    plot_oscillator,
    x1_init=widgets.FloatSlider(value=0.1, min=0.0, max=1.0, step=0.01, description='Initial x1:', continuous_update=True),
    x2_init=widgets.FloatSlider(value=0.0, min=0.0, max=1.0, step=0.01, description='Initial x2:', continuous_update=True)
);